# Evaluate the fine-tuned NLI model on the clean HebNLI test set  ·  Person B

`02_train_nli.ipynb` fine-tuned AlephBERT on contamination-free HebNLI and reported
validation accuracy 0.7667 / macro-F1 0.7668 — the number used *during* training to pick
between configurations. This notebook produces the number that belongs in the report:
accuracy and macro-F1 on `hebnli_test_clean.jsonl`, the held-out split that neither
training nor model selection ever touched.

It only reads. Nothing here calls `.backward()`, nothing here writes into the checkpoint
directory, and the two files under `results/` are the only writes that leave this VM.

**Runs in two places:** the Colab web UI, and VS Code with the Colab extension — same as
the training notebook, and for the same reason: the checkpoint lives on Drive and the GPU
is not local. Run cell by cell, not Run All; each cell says what it should print and what
it writes before it runs.

**Do not point this notebook at `data/probe/review_raw.csv` (689 mined candidates) or
`data/probe/splits/test.jsonl` (151 negation-probe items).** Those measure something
else — the negation probe's test split is held out of NLI *fine-tuning* because it
overlaps the probe, not because it belongs in this evaluation. The only test file this
notebook touches is `data/raw/hebnli_test_clean.jsonl`.

## Where everything ends up

| what | size | where | survives a reset? |
|---|---|---|---|
| cloned repo | ~400 MB | VM disk | no |
| `data/raw/hebnli_test_clean.jsonl` | a few hundred KB | VM disk, regenerated from source | no |
| the checkpoint being evaluated | ~480 MB | **Drive**, read-only | yes — it is already there |
| `results/nli_test_alephbert-hebnli-clean.json`, `results/nli_test_predictions.csv` | a few hundred KB | VM disk, then downloaded | via git |

The checkpoint is the one durable input here: this notebook mounts Drive to read it and
never writes anything back to it.

## 1. Setup

**Prints** &nbsp; Nothing. This cell only defines a helper.

**Writes** &nbsp; Nothing.

In [2]:
# Secrets three ways: Colab's store, then the environment, then a prompt. The VS Code
# extension cannot reach Colab's secret store, so the fallbacks are what make this
# notebook portable. getpass also keeps the token out of the saved output.
import os, subprocess, getpass

def get_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f'{name}: from Colab secrets')
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        print(f'{name}: from environment')
        return value.strip()
    return getpass.getpass(f'{name}: ').strip()

**Prints** &nbsp; Python and torch versions, the GPU name, and which `google.colab`
modules import. `cuda: True` plus a T4 are the two things to confirm before going
further — this notebook does not train, but a forward pass over 883 pairs on CPU is
still slower than it needs to be.

**Writes** &nbsp; Nothing.

In [3]:
# What are we running on? Answers 'will this work here' before anything slow.
import platform
print('python      ', platform.python_version())
print('cwd         ', os.getcwd())
try:
    import torch
    print('torch       ', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu         ', torch.cuda.get_device_name(0))
except ImportError:
    print('torch        not installed yet')
for mod in ('google.colab.userdata', 'google.colab.drive', 'google.colab.files'):
    try:
        __import__(mod)
        print(f'{mod:24s} available')
    except Exception as exc:
        print(f'{mod:24s} NOT available ({type(exc).__name__})')

python       3.12.13
cwd          /content
torch        2.11.0+cu128 | cuda: True
gpu          Tesla T4
google.colab.userdata    available
google.colab.drive       available
google.colab.files       available


**Prints** &nbsp; `GH_TOKEN:` and where it came from, pip's log, then the last 3
commits — the top one should be the newest `nli:` commit.

**Writes** &nbsp; The repo at `/content/hebrew-negation-embeddings` on the VM. The
working directory moves into it, so every path after this is relative to the repo root.

In [ ]:
OWNER, REPO, BRANCH = 'ItayBoros', 'hebrew-negation-embeddings', 'main'

gh_token = get_secret('GH_TOKEN')
url = f'https://{gh_token}@github.com/{OWNER}/{REPO}.git'
bare_url = f'https://github.com/{OWNER}/{REPO}.git'

if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,url,REPO], check=True)
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else '.')

# Re-authenticate origin before every fetch, not just on first clone: the repo is
# private, and the last step below strips the token, so a second run in the same
# session would otherwise fetch unauthenticated and fail.
subprocess.run(['git','remote','set-url','origin', url], check=True)
subprocess.run(['git','fetch','-q','origin',BRANCH], check=True)

# reset --hard, not pull: this VM's checkout is scratch space that always mirrors
# origin, never its own source of truth (every notebook here commits from your
# machine, not from the VM), so local drift must never block picking up a new
# push. `pull` (a merge) refuses whenever an untracked file would be overwritten,
# and scripts in this project routinely create exactly that - writing straight
# into results/ before those paths exist in git - so a later sync hits "untracked
# working tree files would be overwritten by merge" the moment that path is
# committed for real.
subprocess.run(['git','reset','-q','--hard', f'origin/{BRANCH}'], check=True)

# drop the token from the stored remote so it is not left on the VM's disk
subprocess.run(['git','remote','set-url','origin', bare_url], check=True)
del gh_token, url

!pip install -q -r requirements.txt
!git log --oneline -3

**Prints** &nbsp; `all pipeline checks passed`, `all NLI data checks passed`,
`all NLI eval checks passed`. Anything else: stop here, because the numbers downstream
inherit the fault.

**Writes** &nbsp; Nothing.

In [5]:
# offline checks first - seconds, no network, no GPU
!python -m tests.test_data_pipeline | tail -3
!python -m tests.test_nli_data | tail -3
!python -m tests.test_nli_eval | tail -3

wrote 2 rows -> /tmp/tmplg6ujurj/r.csv

all pipeline checks passed
== results table keeps one row per configuration ==

all NLI data checks passed
== check_test_file: the row-count guard ==

all NLI eval checks passed


## 2. Data — the clean test split

`hebnli_test_clean.jsonl` is what `02_train_nli.ipynb` built by running
`src.data.hebnli` then `src.nli.prepare_data --split test`: download HebNLI's test
split, drop the 689 held-out promptIDs, then drop any row whose text matches a probe
sentence surfacing under a *different* promptID. The expected result is fixed:
`884` loaded, `-1` by the id filter, `-0` by the text audit, `883` kept.

`data/raw/` is gitignored and lives only on the VM disk, so a fresh Colab session has to
rebuild it. The cells below check first and only redo the work if the file is missing or
the count is wrong.

**Prints** &nbsp; Whether a clean test file is already on this VM and, if so,
whether its row count is the expected `883`.

**Writes** &nbsp; Nothing.

In [6]:
from pathlib import Path

TEST_CLEAN = Path('data/raw/hebnli_test_clean.jsonl')
n_existing = sum(1 for _ in TEST_CLEAN.open(encoding='utf-8')) if TEST_CLEAN.exists() else 0

NEEDS_REGEN = n_existing != 883
print(f'{TEST_CLEAN}: {n_existing} rows found' if n_existing else f'{TEST_CLEAN}: not found')
print('regeneration needed:', NEEDS_REGEN)

data/raw/hebnli_test_clean.jsonl: not found
regeneration needed: True


**Prints** &nbsp; `HF_TOKEN:` and where it came from — skipped entirely if the
cell above already found a correct clean test file.

**Writes** &nbsp; Nothing. The token stays in memory.

In [7]:
if NEEDS_REGEN:
    os.environ['HF_TOKEN'] = get_secret('HF_TOKEN')
else:
    print('skipped - clean test file already present with the right row count')

**Prints** &nbsp; Skipped entirely if not needed. Otherwise: `rows 884` from the
download, then held-out promptIDs `689`, probe sentences `907`, the three-stage funnel
ending `kept 883`, and `text overlap 0 rows dropped beyond the id filter`.

**Writes** &nbsp; `data/raw/hebnli_test.jsonl` and `data/raw/hebnli_test_clean.jsonl` on
the VM, gitignored.<br>`results/nli_data_test.json` — the manifest, committed, recording
both filter counts.

In [8]:
if NEEDS_REGEN:
    !python -m src.data.hebnli --split test --out data/raw/hebnli_test.jsonl
    !python -m src.nli.prepare_data --source data/raw/hebnli_test.jsonl --split test --out data/raw/hebnli_test_clean.jsonl
else:
    print('skipped - nothing to regenerate')

HebNLI_test.jsonl: 100% 561k/561k [00:00<00:00, 45.9MB/s]
rows                 884
prompts              883
prompts with e/n/c   0
wrote                data/raw/hebnli_test.jsonl

held-out promptIDs   689
probe sentences      907
  loaded                   884
  prompt_id_clean          883  (100% of previous)
  text_clean               883  (100% of previous)

text overlap         0 rows dropped beyond the id filter

wrote                883 rows -> data/raw/hebnli_test_clean.jsonl
manifest             -> results/nli_data_test.json

next: run this once per split, then train with
  python -m src.nli.train_nli --train <train out> --val <val out>


**Prints** &nbsp; The funnel from `results/nli_data_test.json`, then the file's
actual row count. Must read `loaded=884  id_filter=-1  text_audit=-0  kept=883`, and the
row count must be `883` — this is the hard requirement the rest of the notebook depends
on before spending any GPU time.

**Writes** &nbsp; Nothing. It only reads the manifest and the file back.

In [9]:
import json

manifest = json.load(open('results/nli_data_test.json', encoding='utf-8'))
f = manifest['funnel']
print(f"loaded={f['loaded']}  id_filter=-{f['loaded']-f['prompt_id_clean']}"
      f"  text_audit=-{manifest['text_overlap']['rows_dropped']}  kept={manifest['rows_written']}")

n_rows = sum(1 for _ in TEST_CLEAN.open(encoding='utf-8'))
assert n_rows == 883, f'expected exactly 883 rows in {TEST_CLEAN}, found {n_rows}'
assert manifest['rows_written'] == 883
assert f['prompt_id_clean'] < f['loaded'], 'the promptID filter dropped nothing - contamination removal did not run'

print(f'\n{TEST_CLEAN}: {n_rows} rows, contamination-removal filters confirmed applied')
print('clean test file regenerated this run:', NEEDS_REGEN)

loaded=884  id_filter=-1  text_audit=-0  kept=883

data/raw/hebnli_test_clean.jsonl: 883 rows, contamination-removal filters confirmed applied
clean test file regenerated this run: True


## 3. The checkpoint on Drive

**Prints** &nbsp; Drive's mount confirmation, the checkpoint path, and a
confirmation that all four expected files are there. Unlike the training notebook, there
is no VM-disk fallback here: the checkpoint only exists on Drive, so if this cell cannot
mount it, the notebook stops rather than continuing with nothing to evaluate.

**Writes** &nbsp; Nothing beyond mounting Drive at `/content/drive`.

In [10]:
CKPT = '/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    raise RuntimeError(
        'Drive did not mount. The checkpoint only exists on Drive, so evaluation cannot '
        'proceed without it - make sure this session is signed in with the same Google '
        f'account used for training. ({type(exc).__name__}: {exc})'
    ) from exc

ckpt_dir = Path(CKPT)
if not ckpt_dir.is_dir():
    raise RuntimeError(f'{CKPT} not found on Drive - check this mounted the right account')

for name in ('model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json'):
    if not (ckpt_dir / name).exists():
        raise RuntimeError(f'{name} missing from {CKPT}')

print('checkpoint dir:', CKPT)
print('found: model.safetensors, config.json, tokenizer.json, tokenizer_config.json')

Mounted at /content/drive
checkpoint dir: /content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean
found: model.safetensors, config.json, tokenizer.json, tokenizer_config.json


**Prints** &nbsp; The checkpoint path, `encoding pair`, the label names — expect
`{0: 'entailment', 1: 'neutral', 2: 'contradiction'}` — then six pairs each marked `[ok]`
or `[MISMATCH]`, and a tally. This already scored 6/6 once; re-running it here confirms
the mapping still holds for *this* checkpoint before any test-set number is trusted.

**Writes** &nbsp; Nothing.

In [11]:
!python -m src.interventions.check_nli_labels --model {CKPT} --subfolder ""

model     /content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean
encoding  pair
Loading weights: 100% 201/201 [00:07<00:00, 25.52it/s]
labels    {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Expected: entailment
    entailment: 0.835141
       neutral: 0.149420
 contradiction: 0.015438
Predicted: entailment  [ok]

Expected: contradiction
    entailment: 0.002007
       neutral: 0.001679
 contradiction: 0.996314
Predicted: contradiction  [ok]

Expected: neutral
    entailment: 0.016528
       neutral: 0.743961
 contradiction: 0.239511
Predicted: neutral  [ok]

Expected: contradiction
    entailment: 0.002164
       neutral: 0.004595
 contradiction: 0.993241
Predicted: contradiction  [ok]

Expected: contradiction
    entailment: 0.004373
       neutral: 0.004900
 contradiction: 0.990726
Predicted: contradiction  [ok]

Expected: entailment
    entailment: 0.375672
       neutral: 0.321063
 contradiction: 0.303265
Predicted: entailment  [ok]

6/6 agree with the as

## 4. GPU

**Prints** &nbsp; The GPU name and memory. A forward pass over 883 short pairs
takes seconds on any Colab GPU; this only confirms one is attached.

**Writes** &nbsp; Nothing.

In [12]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 5. Evaluate on the clean test set

`src/nli/eval_nli.py` loads the model and tokenizer from `CKPT`, runs a batched forward
pass in `model.eval()` / `no_grad()` mode over `hebnli_test_clean.jsonl`, and writes the
two result files below. It never calls `.backward()` and never writes into `CKPT` —
nothing here can change the weights.

One more guard before running: the test path is asserted to be the clean HebNLI test
split and nothing under `data/probe/`. This notebook must never be pointed at the 689
mined candidates or the 151 negation-probe test items — that is a different measurement,
and running it here would silently invalidate this evaluation's contamination story.

**Prints** &nbsp; The test path being evaluated.

**Writes** &nbsp; Nothing.

In [13]:
TEST_PATH = 'data/raw/hebnli_test_clean.jsonl'
assert TEST_PATH == 'data/raw/hebnli_test_clean.jsonl'
assert 'probe' not in TEST_PATH
print('evaluating on:', TEST_PATH)

evaluating on: data/raw/hebnli_test_clean.jsonl


**Prints** &nbsp; `checkpoint`, `test file`, `examples          883`,
`device cuda`, `pair encoding pair_without_segment_ids`, then accuracy / macro precision
/ macro recall / macro F1 / test loss, a per-class block, and the 3×3 confusion matrix.

**Writes** &nbsp; `results/nli_test_alephbert-hebnli-clean.json` and
`results/nli_test_predictions.csv`, both a few hundred KB, on the VM.

In [14]:
!python -m src.nli.eval_nli --checkpoint {CKPT} \
    --test {TEST_PATH} --expected-n 883 \
    --summary-out results/nli_test_alephbert-hebnli-clean.json \
    --predictions-out results/nli_test_predictions.csv

Loading weights: 100% 201/201 [00:00<00:00, 2744.31it/s]
checkpoint           /content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean
test file            data/raw/hebnli_test_clean.jsonl
examples             883
device               cuda
pair encoding        pair_without_segment_ids

accuracy             0.7961
macro precision      0.7946
macro recall         0.7943
macro F1             0.7940
test loss            0.5115

per-class:
  entailment     precision=0.8100  recall=0.8581  f1=0.8333  support=303
  neutral        precision=0.7426  recall=0.7399  f1=0.7413  support=273
  contradiction  precision=0.8310  recall=0.7850  f1=0.8074  support=307

confusion matrix (rows=true, cols=predicted, order=entailment, neutral, contradiction):
  entailment     [260, 32, 11]
  neutral        [33, 202, 38]
  contradiction  [28, 38, 241]

summary              -> results/nli_test_alephbert-hebnli-clean.json
predictions          -> results/nli_test_predictions.csv


## 6. Inspect the results

**Prints** &nbsp; The summary json pretty-printed, then the first 5 rows of
the predictions csv.

**Writes** &nbsp; Nothing. It only reads the two files back.

In [15]:
import json
import pandas as pd

summary = json.load(open('results/nli_test_alephbert-hebnli-clean.json', encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))

pd.read_csv('results/nli_test_predictions.csv').head()

{
  "checkpoint": "/content/drive/MyDrive/hebrew-negation/checkpoints/alephbert-hebnli-clean",
  "test_file": "data/raw/hebnli_test_clean.jsonl",
  "n_examples": 883,
  "label_ids": {
    "entailment": 0,
    "neutral": 1,
    "contradiction": 2
  },
  "pair_encoding": "pair_without_segment_ids",
  "accuracy": 0.796149490373726,
  "macro_precision": 0.7945501296447249,
  "macro_recall": 0.7943429450508498,
  "macro_f1": 0.7939959737525548,
  "per_class": {
    "entailment": {
      "precision": 0.8099688473520249,
      "recall": 0.858085808580858,
      "f1": 0.8333333333333334,
      "support": 303
    },
    "neutral": {
      "precision": 0.7426470588235294,
      "recall": 0.73992673992674,
      "f1": 0.7412844036697248,
      "support": 273
    },
    "contradiction": {
      "precision": 0.8310344827586207,
      "recall": 0.7850162866449512,
      "f1": 0.8073701842546064,
      "support": 307
    }
  },
  "confusion_matrix": {
    "labels": [
      "entailment",
      "neutra

,pair_id,true_label,predicted_label,prob_entailment,prob_neutral,prob_contradiction,correct
0,112428e,entailment,neutral,0.231355,0.532807,0.235837,False
1,2635e,entailment,entailment,0.922450,0.063207,0.014343,True
2,97156e,entailment,entailment,0.958785,0.030954,0.010261,True
3,75170n,neutral,neutral,0.004467,0.991139,0.004394,True
4,36967e,entailment,entailment,0.913433,0.043732,0.042834,True


## 7. Download the results

**Prints** &nbsp; Two browser downloads, or the files printed inline when
that is unavailable.

**Writes** &nbsp; **Your machine**, finally — as downloads, or as text in the saved
notebook. The checkpoint itself is not among them; it stays on Drive.

In [ ]:
# Browser download is Colab-frontend only, and under the VS Code extension
# `files.download()` can return without raising while the file lands nowhere
# findable - so print the contents unconditionally rather than only on exception;
# that is the one path guaranteed to survive in the saved notebook either way.
RESULTS = ['results/nli_test_alephbert-hebnli-clean.json', 'results/nli_test_predictions.csv']
try:
    from google.colab import files
    for path in RESULTS:
        files.download(path)
except Exception as exc:
    print(f'[warn] browser download unavailable ({type(exc).__name__})')

for path in RESULTS:
    print(f'\n===== {path} =====')
    print(open(path, encoding='utf-8').read())

Commit the two result files from your machine with the `nli:` prefix.

If `data/raw/hebnli_test_clean.jsonl` had to be regenerated this run,
`results/nli_data_test.json` will not have changed from the copy `02_train_nli.ipynb`
already committed — the filters are deterministic, so re-running them changes nothing
worth re-committing.